In [ ]:
import torch
import torch.nn.functional as F
from lightning_definitions import FashionMNISTDataModule, LitModule
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# known precomputed values for the Fashion-MNIST dataset.
# used for normalization
_MEAN = 0.2860
_STD  = 0.3530

In [ ]:
dm = FashionMNISTDataModule(batch_size=32)
dm.prepare_data()
dm.setup()
train_dataloader = dm.train_dataloader()
test_dataloader = dm.test_dataloader()
test_set = test_dataloader.dataset
training_set = train_dataloader.dataset
print(f"Number of training samples: {len(training_set)}")
print(f"Shape of a single sample: {training_set[0][0].shape}, Label: {training_set[0][1]}")
print(f"Number of test samples: {len(test_set)}")


80% of the training set is used for training, that's 48000 samples. 12000 is used for validation.

A single sample has size [1, 28, 28]: a 28x28 pixels image.
Labels are just numbers 0-9.

Test set is 10000 samples. We don't touch it until the final tests.

In [ ]:
# Visualize a batch of images

# Create iterator once
test_iter = iter(dm.test_dataloader())

In [ ]:
# Each time this cell is re-run, new images appear.
# Get next batch
images, labels = next(test_iter)
images = images.cpu()
labels = labels.cpu()


# Plot first 10 images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for ax, img, label in zip(axes.flatten(), images[:10], labels[:10]):
    # Undo normalization
    img = img * _STD + _MEAN

    # Convert [1, 28, 28] -> [28, 28]
    img = img.squeeze()

    ax.imshow(img, cmap="gray")
    ax.set_title(str(label.item()))
    ax.axis("off")

plt.tight_layout()
plt.show()

This is what the actual images look like, and their corresponding classes.

Some classes look pretty similar, like pullover vs coat, sneaker vs ankle boot, etc.

In [ ]:
targets = []
for pair in training_set:
    targets.append(pair[1])
targets = np.array(targets)
unique, counts = np.unique(targets, return_counts=True)
print("Class distribution in the training set:")
for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count} samples")

The training dataset is very well balanced.

Old checkpoints can still be loaded after adding helper methods like `predict()` to `LitModule` because Lightning serializes the model weights and hyperparameters, not the method definitions.

In [ ]:
model = LitModule.load_from_checkpoint(
    "/home/denis/Coding/nas-fashion-mnist/lightning_logs/mlp_basic/checkpoints/epoch=19-step=7500.ckpt",
    map_location=torch.device("cpu")
)
model.eval()

# Single image prediction (grayscale [H, W])
single_image = images[0].squeeze(0)
label, confidence = model.predict(single_image)
print(f"Single image -> class={label}, confidence={confidence:.4f}")

# Batch prediction (grayscale [B, H, W])
batch_images = images[:10].squeeze(1)
batch_results = model.predict(batch_images)
print("Batch results:")
for idx, (cls, prob) in enumerate(batch_results):
    print(f"  image {idx}: class={cls}, confidence={prob:.4f}")


In [ ]:
model = LitModule.load_from_checkpoint(
    "/home/denis/Coding/nas-fashion-mnist/lightning_logs/mlp_basic/checkpoints/epoch=19-step=7500.ckpt",
    map_location=torch.device("cpu")
)
model.eval()

# Single image prediction (grayscale [H, W])
single_image = images[0].squeeze(0)
label, confidence = model.predict(single_image)
print(f"Single image -> class={label}, confidence={confidence:.4f}")

# Batch prediction (grayscale [B, H, W])
batch_images = images[:10].squeeze(1)
batch_results = model.predict(batch_images)
print("Batch results:")
for idx, (cls, prob) in enumerate(batch_results):
    print(f"  image {idx}: class={cls}, confidence={prob:.4f}")

In [ ]:
images[0].shape

In [ ]:
random_image = torch.rand(1, 28, 28)
random_image.shape

In [ ]:
image = images[0].to(model.device)
image.device

In [ ]:
# TODO: make this into a predict method in the LitModule class, so that we can just call model.predict(image)